<a href="https://colab.research.google.com/github/joserlandero/DatosMasivos/blob/main/Tarea_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea 1
---



## Instalación de spark

In [3]:
! pip install spark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.7/351.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.8 MB/s eta 0:00:00


In [5]:
import spark
spark

<module 'spark' from '/usr/local/lib/python3.12/dist-packages/spark/__init__.py'>

## Dataset

Se usará un conjunto de datos de datos sobre siniestros de automoviles en estados unidos

## ¿Por qué?
En el ambito de los seguros, es importante el conocer cuanto costarán los siniestros. Para esto existen metodos tradicionales que pueden ayudar a resolver el problema. Con el boom del machine learning y Deep Learning, se han abierto las posibilidades a encontrar una nueva metodología que supere las tradicionales.

**Spark** será de gran ayuda para el manejo de grandes volumenes de datos ya que el registro de siniestros es muy grande

In [ ]:
import numpy as np

In [6]:
path = "/content/Austin_Crash_Report_Data_-_Crash_Level_Records.csv"

In [9]:
#Creamos sesion de spark
# Usamos librerias necesarias
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

#Usamos time para probar tiemos
import time

inicio = time.perf_counter()

spark = (SparkSession.builder.appName("Siniestros").getOrCreate())

df = spark.read.csv(path, header=True, inferSchema=True)

fin = time.perf_counter()

In [13]:
display(df.limit(5).toPandas())
print(f"Tiempo de spark: {fin-inicio}")

,ID,Crash ID,crash_fatal_fl,case_id,rpt_block_num,rpt_street_name,rpt_street_sfx,crash_speed_limit,road_constr_zone_fl,latitude,...,Is deleted,Is temporary record,Law enforcement fatality count,Reported street prefix,Estimated Maximum Comprehensive Cost,Estimated Total Comprehensive Cost,Location ID,Location group,Address,Collision type
0,1417,13668262,False,140141596,None,ANTHONY,ST,30,False,NaN,...,False,False,0,None,20000.0,40000.0,None,NaN,ANTHONY ST & HOLLY ST,None
1,2706,13699891,False,140510958,8900,NOT REPORTED,BLVD,45,False,NaN,...,False,False,0,None,20000.0,40000.0,None,NaN,8900 US 183 SVRD,None
2,2907,13704955,False,140410151,None,NOT REPORTED,HWY,45,False,NaN,...,False,False,0,None,20000.0,20000.0,None,NaN,IH 35,None
3,1099,13662431,False,140121219,4900,NOT REPORTED,HWY,-1,False,NaN,...,False,False,0,None,20000.0,40000.0,None,NaN,4900 IH 35,None
4,976,13658778,False,140260929,1200,ROBERT E LEE,DR,35,False,NaN,...,False,False,0,None,250000.0,250000.0,None,NaN,1200 ROBERT E LEE DR,None


Tiempo de spark: 27.993247524000026


In [16]:
# Con schema imprimimos los tipos de datos
df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Crash ID: integer (nullable = true)
 |-- crash_fatal_fl: boolean (nullable = true)
 |-- case_id: string (nullable = true)
 |-- rpt_block_num: string (nullable = true)
 |-- rpt_street_name: string (nullable = true)
 |-- rpt_street_sfx: string (nullable = true)
 |-- crash_speed_limit: integer (nullable = true)
 |-- road_constr_zone_fl: boolean (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- crash_sev_id: integer (nullable = true)
 |-- sus_serious_injry_cnt: integer (nullable = true)
 |-- nonincap_injry_cnt: integer (nullable = true)
 |-- poss_injry_cnt: integer (nullable = true)
 |-- non_injry_cnt: integer (nullable = true)
 |-- unkn_injry_cnt: integer (nullable = true)
 |-- tot_injry_cnt: integer (nullable = true)
 |-- death_cnt: integer (nullable = true)
 |-- units_involved: string (nullable = true)
 |-- contro: string (nullable = true)
 |-- point: string (nullable = true)
 |-- motor_ve

In [19]:
# Hacemos o manipulamos el dataset
#Por ejemplo filtramos mas de dos muertos
df.filter(df.death_cnt>2).show()

+-------+--------+--------------+---------+-------------+---------------+--------------+-----------------+-------------------+-----------+------------+------------+---------------------+------------------+--------------+-------------+--------------+-------------+---------+--------------------+------+--------------------+-------------------------+----------------------------------+-------------------+----------------------------+----------------------+-------------------------------+----------------------+-------------------------------+-----------------+--------------------------+--------+-------------+----------------------------------+-------------------------+----------------------------+--------------------+----------+-------------------+------------------------------+----------------------+------------------------------------+----------------------------------+-----------+--------------+--------------------+--------------+
|     ID|Crash ID|crash_fatal_fl|  case_id|rpt_block_num|r

In [20]:
# Filtramos costos mayores a 1 millon
from pyspark.sql.functions import col
df.filter(col('Estimated Maximum Comprehensive Cost')>1000000).show()

+------+--------+--------------+---------+-------------+-------------------+--------------+-----------------+-------------------+-----------+------------+------------+---------------------+------------------+--------------+-------------+--------------+-------------+---------+--------------------+------+--------------------+-------------------------+----------------------------------+-------------------+----------------------------+----------------------+-------------------------------+----------------------+-------------------------------+-----------------+--------------------------+--------+-------------+----------------------------------+-------------------------+----------------------------+--------------------+----------+-------------------+------------------------------+----------------------+------------------------------------+----------------------------------+-----------+--------------+--------------------+--------------+
|    ID|Crash ID|crash_fatal_fl|  case_id|rpt_block_num

In [21]:
# Como en pandas se pueden ver o obtener informacion de las variables
df.describe().show()

+-------+------------------+--------------------+--------------------+------------------+------------------+--------------+------------------+------------------+-------------------+------------------+---------------------+-------------------+------------------+------------------+-------------------+------------------+--------------------+--------------------+------+--------------------+-------------------------+----------------------------------+--------------------+----------------------------+----------------------+-------------------------------+----------------------+-------------------------------+-----------------+--------------------------+----------------------------------+-------------------------+----------------------------+--------------------+------------------------------+----------------------+------------------------------------+----------------------------------+-----------+------------------+--------------------+--------------+
|summary|                ID|            

In [23]:
# Con withcolumns podemos hacer operaciones de columnas y crear una nueva

df.withColumn('Total de muertes', df.death_cnt+df.other_death_count).show()

+----+--------+--------------+---------+-------------+---------------+--------------+-----------------+-------------------+---------+----------+------------+---------------------+------------------+--------------+-------------+--------------+-------------+---------+--------------------+------+--------------------+-------------------------+----------------------------------+-------------------+----------------------------+----------------------+-------------------------------+----------------------+-------------------------------+-----------------+--------------------------+--------+-------------+----------------------------------+-------------------------+----------------------------+--------------------+----------+-------------------+------------------------------+----------------------+------------------------------------+----------------------------------+-----------+--------------+--------------------+--------------+----------------+
|  ID|Crash ID|crash_fatal_fl|  case_id|rpt_bloc